# Agent live observability: trace the Aria RM briefing agent with OpenTelemetry

This notebook traces a Foundry-hosted agent using the Azure AI Projects SDK and Azure Monitor. The example agent is **`aria-rm-briefing-agent`** - the intent-level MCP briefing assistant for Contoso Private Investments relationship managers, created in [08-05b-01-private-banking-agent-setup.ipynb](../08-05b-contoso-private-banking-mcp/08-05b-01-private-banking-agent-setup.ipynb).

Aria is interesting for an observability lab because it uses a **hosted MCP tool** rather than a synthetic function tool. The MCP server (`contoso_private_banking`) runs server-side in Foundry, so the trace we capture reflects real tool execution - not stubbed-out function-call IDs that never get resolved.

## Prerequisites

1. **Python environment**: Run `uv sync` from the repository root to create the shared `.venv`, then select the `.venv` kernel in VS Code.
2. **Run [`08-07-01-deploy-observability-infra.ipynb`](08-07-01-deploy-observability-infra.ipynb) first** - deploys Application Insights and writes the `.env` keys this notebook reads.
3. **Run [`08-05b-01-private-banking-agent-setup.ipynb`](../08-05b-contoso-private-banking-mcp/08-05b-01-private-banking-agent-setup.ipynb) first** - creates `aria-rm-briefing-agent` on the admin project. This notebook fails fast if the agent is missing.

The following `.env` keys must be present:

| Key | Set by |
|-----|---------|
| `OBS_APP_INSIGHTS_CONN_STRING` | `08-07-01-deploy-observability-infra.ipynb` |
| `OBS_APP_INSIGHTS_NAME` | `08-07-01-deploy-observability-infra.ipynb` |
| `OBS_RESOURCE_GROUP` | `08-07-01-deploy-observability-infra.ipynb` |

## What you'll learn

| Concept | Description |
|---------|-------------|
| **SDK Instrumentation** | Enable tracing with `AIProjectInstrumentor` |
| **OpenTelemetry Setup** | Configure Azure Monitor exporter |
| **MCP-tool spans** | What trace shape a hosted-MCP agent produces vs a function-tool agent |
| **Automatic Tracing** | Traces captured for all SDK operations including MCP tool calls |
| **Trace Verification** | Query App Insights via KQL to verify trace arrival |

## References

- [Agent Tracing Overview](https://learn.microsoft.com/azure/ai-foundry/observability/concepts/trace-agent-concept?view=foundry)
- [Tracing Integrations](https://learn.microsoft.com/azure/ai-foundry/observability/how-to/trace-agent-framework?view=foundry)

In [1]:
import os, subprocess, hashlib, time
from pathlib import Path
from dotenv import load_dotenv

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

# OTel exporter destination (deployed by 08-07-01)
OBS_APP_INSIGHTS_CONN_STRING = os.environ['OBS_APP_INSIGHTS_CONN_STRING']
OBS_APP_INSIGHTS_NAME        = os.environ['OBS_APP_INSIGHTS_NAME']
OBS_RESOURCE_GROUP           = os.environ['OBS_RESOURCE_GROUP']

# NOTE: We deliberately do NOT set AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING=true.
# That env var, combined with `enable_content_recording=True` in the
# instrumentor (Step 2), triggers a content-capture code path in the OpenAI
# client that the admin project's responses endpoint rejects with
# `401 Token not supported`. The pure span-level tracing path works fine.
# Trade-off: `gen_ai.event.content` events (request/response bodies in the
# `traces` table) won't be captured - spans in `dependencies` still are.

# Admin project endpoint - same SUFFIX derivation as 08-05b. The Aria agent
# we trace lives on the admin project (not on a spoke), because its parent
# account is where the MCP-tool agent was registered.
SUBSCRIPTION_ID  = subprocess.run(
    'az account show --query id -o tsv', shell=True, capture_output=True, text=True
).stdout.strip()
SUFFIX           = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'

AGENT_NAME = 'aria-rm-briefing-agent'

print('Configuration loaded')
print(f'  Project endpoint:   {PROJECT_ENDPOINT}')
print(f'  App Insights:       {OBS_APP_INSIGHTS_NAME}')
print(f'  Agent:              {AGENT_NAME}')

Configuration loaded
  Project endpoint:   https://aif-core-c2676f.services.ai.azure.com/api/projects/project-admin-c2676f
  App Insights:       appi-obs-n5d3ja
  Agent:              aria-rm-briefing-agent


---
## Step 1: Configure OpenTelemetry with Azure Monitor

The `configure_azure_monitor()` function sets up all OpenTelemetry components automatically:
- TracerProvider with Azure Monitor exporter
- MetricProvider (optional)
- LoggerProvider (optional)

This must be called **before** instrumenting the SDK.

In [2]:
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import trace as otel_trace
from opentelemetry.sdk.trace import TracerProvider

# Check if already configured
current_provider = otel_trace.get_tracer_provider()
if isinstance(current_provider, TracerProvider):
    provider = current_provider
    print('Reusing existing TracerProvider')
else:
    configure_azure_monitor(connection_string=OBS_APP_INSIGHTS_CONN_STRING)
    provider = otel_trace.get_tracer_provider()
    print('OpenTelemetry configured with Azure Monitor')

print('TracerProvider ready')

OpenTelemetry configured with Azure Monitor
TracerProvider ready


---
## Step 2: Instrument Azure AI Projects SDK

The SDK must be **explicitly instrumented** to enable tracing:

```python
from azure.ai.projects.telemetry import AIProjectInstrumentor
AIProjectInstrumentor().instrument(enable_content_recording=False)
```

This must happen **before** creating any clients.

> **Note on `enable_content_recording`** - we set this to `False` to work around a 401 on the admin project's responses endpoint when content recording is on. Spans (chat, responses, MCP tool spans) are still captured fully in the `dependencies` table - only the request/response body events in the `traces` table are skipped. Against a spoke project with `enable_content_recording=True`, this lab's original NASA version captured spans but the content events table came back empty anyway, so the practical loss is minimal.

In [3]:
from azure.ai.projects.telemetry import AIProjectInstrumentor

AIProjectInstrumentor().instrument(enable_content_recording=False)

print('Azure AI Projects SDK instrumented')
print('  Content recording: disabled (spans still captured - see note in Step 2)')

Azure AI Projects SDK instrumented
  Content recording: disabled (spans still captured — see note in Step 2)


---
## Step 3: Create Foundry client

Now that instrumentation is enabled, all SDK operations will be automatically traced.

In [4]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

# Initialize the Foundry client (now instrumented)
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

# Get OpenAI client for Responses API
openai_client = project_client.get_openai_client()

print('Connected to Foundry admin project')
print(f'  Endpoint: {PROJECT_ENDPOINT[:60]}...')

Connected to Foundry admin project
  Endpoint: https://aif-core-c2676f.services.ai.azure.com/api/projects/p...


---
## Step 4: Reference the Aria briefing agent

We're not creating a new agent - `aria-rm-briefing-agent` was created on the admin project by [08-05b](../08-05b-contoso-private-banking-mcp/08-05b-01-private-banking-agent-setup.ipynb) with its MCP tool already attached. We just look up the latest version.

This means the `create_agent` span you might expect from the SDK won't appear in the trace below - only invocation spans (`responses`, `chat`, and MCP tool execution) will be captured.

> If the agent doesn't exist, the cell below raises with a pointer back to 08-05b. We deliberately don't auto-create - keeps the dependency explicit and avoids divergence from the real 08-05b agent.

In [5]:
from azure.core.exceptions import ResourceNotFoundError

try:
    existing_versions = list(project_client.agents.list_versions(agent_name=AGENT_NAME))
except ResourceNotFoundError:
    existing_versions = []

if not existing_versions:
    raise RuntimeError(
        f"Agent '{AGENT_NAME}' not found on the admin project.\n"
        f"  Run 08-05b-01-private-banking-agent-setup.ipynb first to create it."
    )

agent         = existing_versions[0]
AGENT_VERSION = agent.version

print(f'Found agent: {agent.name} v{AGENT_VERSION}')
print(f'  MCP tool already attached (set up in 08-05b).')

Found agent: aria-rm-briefing-agent v1
  MCP tool already attached (set up in 08-05b).


---
## Step 5: Invoke the agent (automatically traced)

When we invoke Aria, the SDK captures:
- The outer `responses` span (one per query)
- The inner `chat` spans Foundry makes against the chat completion model
- MCP tool invocations against the `contoso_private_banking` server (tool name, server label, latency)
- Errors and tool failures

We send three queries that exercise three different intent-level MCP tools - the full briefing flow (`cpb_prepare_client_briefing`), an activity summary (`cpb_summarize_recent_activity`), and a research lookup (`cpb_find_relevant_research`).

In [6]:
queries = [
    'Prepare my morning briefing for the meeting with the Berger Family Trust (cli-001).',
    'What recent activity should I know about for the Lindemann Family Office (cli-003)?',
    'Find me research relevant to ESG mandates and European bank capital rules.',
]

print('Invoking Aria RM briefing agent (traces captured automatically)...\n')

for query in queries:
    print(f'User: {query}')

    start = time.time()
    response = openai_client.responses.create(
        input=query,
        extra_body={
            'agent_reference': {
                'name': AGENT_NAME,
                'version': AGENT_VERSION,
                'type': 'agent_reference'
            }
        },
    )
    duration = (time.time() - start) * 1000

    output_text = getattr(response, 'output_text', None) or str(response.output)
    print(f'Aria: {str(output_text)[:300]}')
    print(f'  Duration: {duration:.0f}ms\n')

    time.sleep(1)

print('Agent invocations complete!')

Invoking Aria RM briefing agent (traces captured automatically)...

User: Prepare my morning briefing for the meeting with the Berger Family Trust (cli-001).
Aria: Here is your morning briefing for the meeting with the Berger Family Trust (client cli-001):

Client Overview:
- Name: Berger Family Trust
- Segment: UHNW Multi-Generation
- Relationship Manager: Anna Müller
- Base Currency: CHF
- Assets Under Management (AUM): CHF 89,533,698 (calculated)
- Next rev
  Duration: 25416ms

User: What recent activity should I know about for the Lindemann Family Office (cli-003)?
Aria: For the Lindemann Family Office (cli-003), recent activity over the past 30 days includes three transactions:

1. A thematic AI addition purchase on 6 May 2026, acquiring 5,000 units at a price of 213.1 USD, amounting to 948,000 CHF.
2. A private equity capital call on 28 April 2026, with an amount 
  Duration: 20825ms

User: Find me research relevant to ESG mandates and European bank capital rules.
Aria: Here are 

---
## Step 6: Flush and verify traces

OpenTelemetry batches traces for efficiency. We flush to ensure all traces are sent, then verify arrival in App Insights.

In [10]:
# Force flush all pending traces
print('Flushing traces to Application Insights...')
provider.force_flush()
print('Traces flushed\n')

# Wait for ingestion (App Insights has ~30 second delay)
print('Waiting 45 seconds for trace ingestion...')
time.sleep(45)

Flushing traces to Application Insights...
Traces flushed

Waiting 45 seconds for trace ingestion...


In [9]:
from datetime import timedelta
from azure.monitor.query import LogsQueryClient
from azure.identity import DefaultAzureCredential

print(f'Querying Application Insights: {OBS_APP_INSIGHTS_NAME}\n')

logs_client = LogsQueryClient(credential=DefaultAzureCredential())
resource_id = (
    f'/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{OBS_RESOURCE_GROUP}'
    f'/providers/microsoft.insights/components/{OBS_APP_INSIGHTS_NAME}'
)

# Looking for chat/responses/MCP-tool spans. We do NOT look for `create_agent`
# here - Aria was created by 08-05b, so no agent-creation span appears in this
# session. The `mcp`/`tool` filters surface server-side MCP tool execution.
# Content events (gen_ai.event.content in the traces table) are not queried -
# they require enable_content_recording=True, which we disabled in Step 2.
# Columns: 0=timestamp 1=operation_Id 2=span 3=duration_ms 4=success
#          5=model 6=input_tokens 7=output_tokens 8=system 9=agent
span_query = """
dependencies
| where timestamp > ago(30m)
| where name contains "chat" or name contains "responses" or name contains "mcp" or name contains "tool"
| project
    timestamp,
    operation_Id,
    span          = name,
    duration_ms   = duration,
    success,
    model         = tostring(customDimensions['gen_ai.request.model']),
    input_tokens  = toint(customDimensions['gen_ai.usage.input_tokens']),
    output_tokens = toint(customDimensions['gen_ai.usage.output_tokens']),
    system        = tostring(customDimensions['gen_ai.system']),
    agent         = tostring(customDimensions['agent.name'])
| order by timestamp asc
"""

try:
    span_resp = logs_client.query_resource(resource_id, span_query, timespan=timedelta(minutes=30))

    rows = span_resp.tables[0].rows if span_resp.tables else []
    if rows:
        print(f"{'Timestamp':<10} {'Span':<42} {'Model / Agent':<35} {'In':>6} {'Out':>6} {'ms':>7}  Status")
        print('─' * 118)
        for r in rows:
            ts     = str(r[0])[11:19] if r[0] else 'N/A'
            span   = str(r[2])[:40]
            model  = (str(r[5]) if r[5] else str(r[9]) if r[9] else '-')[:33]
            inp    = str(r[6]) if r[6] is not None else '-'
            out    = str(r[7]) if r[7] is not None else '-'
            ms     = f'{r[3]:.0f}' if r[3] else '-'
            status = 'OK' if r[4] else 'FAIL'
            print(f'{ts:<10} {span:<42} {model:<35} {inp:>6} {out:>6} {ms:>7}  {status}')
    else:
        print('No spans found in last 30 minutes - wait 30-45 seconds and retry')

except Exception as e:
    import traceback
    traceback.print_exc()


Querying Application Insights: appi-obs-n5d3ja

No spans found in last 30 minutes — wait 30–45 seconds and retry


---
## Step 7: View your traces

Now that traces have been captured, let's generate direct links to view them in both the Foundry Portal and Azure Portal.